# Lilylet NotaGen — INT8 ONNX-Runtime autoregressive generation

Loads the INT8-quantized ONNX weights (`patch_int8.onnx` + `token_int8.onnx`, exported by `tools/export_lilylet_int8_ort.py`) and runs hierarchical patch/token autoregressive generation through ONNX Runtime instead of PyTorch.

The `ORTGenerator` (from `tests/bench_lilylet_int8_ort.py`) mirrors `LilyletPatchyGenerator.generate`: the two heavy transformer forwards run as ORT sessions, while the cheap embedding lookup / patch-state splice / sampling stay in numpy/torch. A torch `LilyletPatchyGenerator` is still built — only for the tokenizer and the patch_size / bos / eos ids.

In [1]:
from pathlib import Path
import sys
import os
import time

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
# ORTGenerator lives in the tests/ bench module
if str(REPO_ROOT / 'tests') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'tests'))

import torch
from starry.utils.config import Configuration
from starry.lilylet.patchyGenerator import LilyletPatchyGenerator
from bench_lilylet_int8_ort import ORTGenerator

# The run whose int8 onnx we exported. RUN dir holds best.chkpt, .state.yaml and onnx/.
RUN = os.path.expanduser('~/data/models/deep-starry-logs/lilylet/20260611-lilylet-notagenx-1m0611-llama')
CKPT = os.path.join(RUN, 'best.chkpt')
ONNX_DIR = os.path.join(RUN, 'onnx')
PATCH_INT8 = os.path.join(ONNX_DIR, 'patch_int8.onnx')
TOKEN_INT8 = os.path.join(ONNX_DIR, 'token_int8.onnx')
THREADS = 14

TOKENIZER = str(REPO_ROOT / 'assets' / 'lilylet-tokenizer.json')
assert os.path.isfile(PATCH_INT8) and os.path.isfile(TOKEN_INT8), 'run tools/export_lilylet_int8_ort.py first'
print('run :', RUN)
print('int8:', os.path.getsize(PATCH_INT8)/1e6, 'MB +', os.path.getsize(TOKEN_INT8)/1e6, 'MB')

run : /home/camus/data/models/deep-starry-logs/lilylet/20260611-lilylet-notagenx-1m0611-llama
int8: 393.874949 MB + 117.035407 MB


In [2]:
# Build a torch generator from the run's config (architecture from .state.yaml).
# It loads the fp32 checkpoint too, but we only use it for the tokenizer + the
# patch_size / bos / eos / pad ids; all heavy forwards go through ORT below.
torch.set_num_threads(THREADS)
config = Configuration.createOrLoad(RUN, volatile=True)
gen = LilyletPatchyGenerator.from_config(config, CKPT, tokenizer_path=TOKENIZER, device='cpu')
print('base_type:', config['model.args.base_type'], '| patch_size:', gen.patch_size)
print('pad/bos/eos:', gen.pad_id, gen.bos_id, gen.eos_id)

# Wrap the int8 ONNX sessions. This is the actual inference engine.
ort_gen = ORTGenerator(gen, PATCH_INT8, TOKEN_INT8, threads=THREADS)
print('ORT int8 sessions ready')

base_type: llama | patch_size: 16
pad/bos/eos: 0 1 2
ORT int8 sessions ready


In [3]:
# Sanity: encode a protected token and decode it back through the tokenizer.
demo = gen.tokenizer.encode('[r:0/8]')
print('encode "[r:0/8]" ->', demo)
print('decode back        ->', repr(gen.patch_to_text(demo)))

encode "[r:0/8]" -> [91, 114, 58, 48, 47, 56, 93]
decode back        -> '[r:0/8]'


In [4]:
# Conditional generation: seed a metadata header and let the int8 model continue.
# (ORTGenerator.generate mirrors the torch generate signature, minus `verbose`.)
PROMPT = '[composer "Schubert, Franz"]\n[genre "Romantic"]\n[instrument "Keyboard"]\n'
torch.manual_seed(0)
t0 = time.perf_counter()
text = ort_gen.generate(prompt_text=PROMPT, max_patches=1024,
                        temperature=0.9, top_k=20, top_p=0.95,
                        measures=8, postprocess=True)
dt = time.perf_counter() - t0
print(text)
print('\n===== %d chars, %d lines in %.1fs =====' % (len(text), text.count('\n') + 1, dt))

[composer "Schubert, Franz"]
[genre "Romantic"]
[instrument "Keyboard"]

\staff "1" \key g \major \major \time 2/4 \clef "treble" \tempo 4=132 ^\markup "Allegro" b'16\pp( c \\
\staff "2" \clef "bass" r8 | % r:0/8

\staff "1" \key g \major \major \time 2/4 d'8-. b)-. g-. b-. \\
\staff "2" <g, g'>8-. <b' d>-. <g b>-. <b d>-. | % r:1/7

\staff "1" \key g \major \time 2/4 g'8-. d-. d16( c b a \\
\staff "2" <g b>8-. <g b>-. <g b>-. r | % r:2/69

\staff "1" \key g \major \time 2/4 g8-. g)-. d'16( cs b cs \\
\staff "2" <g,, g'>8-. <g' d'>-. <g d'>-. <g d'>-. | % r:3/68

\staff "1" \key g \major \time 2/4 d8-. d-. d16( cs b cs \\
\staff "2" <g, d'>8-. <g d'>-. <g d'>-. r | % r:4/67

\staff "1" \key g \major \time 2/4 d8-. d)-. d'16( cs b cs \\
\staff "2" <d,, d'>8-. <d' a'>-. <d a'>-. <d a'>-. | % r:5/66

\staff "1" \key g \major \time 2/4 d'8\<-. d-. d16( cs b cs \\
\staff "2" <d, a'>8-. <d a'>-. <d a'>-. r | % r:6/65

\staff "1" \key g \major \time 2/4 d'8\f-. d16( e fs e d cs \\
\staff "2" 

In [5]:
# Save the generated piece to a .lyl file.
out_dir = REPO_ROOT / 'tests' / 'output'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'lilylet_onnx_int8_generated.lyl'
out_path.write_text(text)
print('wrote', out_path)

wrote /home/camus/work/deep-starry/tests/output/lilylet_onnx_int8_generated.lyl
